In [1]:
# conda activate anndata

import os
import sys
import anndata as ad

sys.path.append("/mnt/lareaulab/reliscu/code")

from junction2psi import *

In [2]:
adata = ad.read_h5ad("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/frontal_cortex/GTEx_frontal_cortex_SJ_counts.h5ad")

In [5]:
SJ_counts_table = pd.DataFrame(adata.X.T, columns=adata.obs_names, index=adata.var_names)
SJ_counts_table.shape

(233295, 269)

In [3]:
# Subset to samples kept after QC

In [4]:
expr = pd.read_csv("GTEx_frontal_cortex_counts_TMMF_SampleNetworks/All_10-11-36/GTEx_frontal_cortex_counts_TMMF_All_247_outliers_removed.csv", index_col=0)
expr.columns = expr.columns.str.replace(".","-")

In [6]:
SJ_counts_table = SJ_counts_table[expr.columns]
SJ_counts_table.shape

(233295, 247)

In [ ]:
events_i1 = pd.Index([x[:-3] for x in SJ_counts_table.index if '_I1' in x])
events_i2 = pd.Index([x[:-3] for x in SJ_counts_table.index if '_I2' in x])
events_se = pd.Index([x[:-3] for x in SJ_counts_table.index if '_SE' in x])

events = events_i1.intersection(events_i2).intersection(events_se)
i1_events = [x + '_I1' for x in events]
I1_table = SJ_counts_table.loc[i1_events]
I1_table.index = events

i2_events = [x + '_I2' for x in events]
I2_table = SJ_counts_table.loc[i2_events]
I2_table.index = events

se_events = [x + '_SE' for x in events]
SE_table = SJ_counts_table.loc[se_events]
SE_table.index = events
    
I1_filt = I1_table.index[I1_table.sum(axis=1) > 0]
I2_filt = I2_table.index[I2_table.sum(axis=1) > 0]
SE_filt = SE_table.index[SE_table.sum(axis=1) > 0]

filtered_events = I1_filt.intersection(I2_filt).intersection(SE_filt)

I1_table = I1_table.loc[filtered_events]
I2_table = I2_table.loc[filtered_events]
SE_table = SE_table.loc[filtered_events]

psi = ((I1_table + I2_table) /(2*SE_table + I1_table + I2_table)).fillna(0)
reads = SE_table + I1_table + I2_table

In [26]:
psi.shape[0]

27126

In [20]:
psi.head()

,GTEX-1117F-0011-R10b-SM-GI4VE,GTEX-111FC-0011-R10a-SM-GIN8G,GTEX-117XS-0011-R10b-SM-GIN8Z,GTEX-1192W-0011-R10b-SM-GHWOF,GTEX-1192X-0011-R10a-SM-DO941,GTEX-11DXW-0011-R10b-SM-GI4VP,GTEX-11DXY-0011-R10b-SM-DO12C,GTEX-11DYG-0011-R10b-SM-DNZZO,GTEX-11DZ1-0011-R10b-SM-DO943,GTEX-11EI6-0011-R10a-SM-DO93R,...,GTEX-ZDXO-0011-R10a-SM-4WWD8,GTEX-ZF28-0011-R10a-SM-4WWEH,GTEX-ZUA1-0011-R10a-SM-51MT6,GTEX-ZV68-0011-R10a-SM-51MT7,GTEX-ZVT3-0011-R10b-SM-57WB6,GTEX-ZVZQ-0011-R10b-SM-51MRT,GTEX-ZXG5-0011-R10a-SM-57WDD,GTEX-ZYFD-0011-R10a-SM-GPI91,GTEX-ZYY3-0011-R10a-SM-GNTAZ,GTEX-ZZPT-0011-R10b-SM-GPI8B
ENSG00000292994_other_1,1.0,1.000000,0.666667,1.0,0.0,0.090909,0.750000,0.25,0.0,0.0,...,0.2,0.875000,1.0,0.5,1.000000,1.000000,1.0,1.000000,1.0,1.0
ENSG00000290385_other_1,0.0,0.272727,0.333333,1.0,0.0,0.000000,0.200000,0.00,0.0,0.0,...,1.0,0.142857,0.0,0.0,0.090909,0.142857,0.0,0.000000,0.0,0.0
ENSG00000290385_other_2,0.0,0.272727,0.200000,1.0,0.0,0.076923,0.333333,0.00,0.0,0.0,...,1.0,0.000000,0.0,1.0,0.230769,0.142857,0.0,0.142857,0.0,0.0
ENSG00000290385_other_3,1.0,1.000000,0.428571,0.0,1.0,0.777778,0.500000,0.00,1.0,0.0,...,1.0,1.000000,0.5,0.0,1.000000,1.000000,1.0,1.000000,1.0,1.0
ENSG00000237491_other_7,1.0,1.000000,0.000000,1.0,1.0,0.000000,1.000000,1.00,1.0,1.0,...,1.0,1.000000,1.0,0.0,0.000000,0.000000,0.0,0.000000,1.0,1.0


In [27]:
psi.to_csv(f"data/GTEx_cortex_SJ_PSI.csv")

In [ ]:
# reads.to_csv(f"data/GTEx_cortex_SJ_counts.csv")